# LeetCode 450: Delete Node in a BST

**Difficulty**: Medium  
**Topics**: Tree, Binary Search Tree (BST), Binary Tree, Recursion  
**Link**: [LeetCode Problem](https://leetcode.com/problems/delete-node-in-a-bst/)

---

## Problem Statement

Given a root node reference of a BST and a key, delete the node with the given key in the BST. Return the root node reference (possibly updated) of the BST.

Basically, the deletion can be divided into two stages:
1. Search for a node to remove
2. If the node is found, delete the node

### Visual Example

```
Original BST:        Delete 3:
      5                  5
     / \                / \
    3   6              4   6
   / \   \            /     \
  2   4   7          2       7

Node 3 has two children, so we replace it with its
inorder successor (4) or predecessor (2).
```

### Examples

**Example 1:**
```
Input: root = [5,3,6,2,4,null,7], key = 3
Output: [5,4,6,2,null,null,7]
Explanation: Given key to delete is 3. So we find the node with value 3 and delete it.
One valid answer is [5,4,6,2,null,null,7].
Another valid answer is [5,2,6,null,4,null,7].
```

**Example 2:**
```
Input: root = [5,3,6,2,4,null,7], key = 0
Output: [5,3,6,2,4,null,7]
Explanation: The tree does not contain a node with value = 0.
```

**Example 3:**
```
Input: root = [], key = 0
Output: []
```

### Constraints

- The number of nodes in the tree is in the range `[0, 10^4]`
- `-10^5 <= Node.val <= 10^5`
- Each node has a **unique** value
- `root` is a valid binary search tree
- `-10^5 <= key <= 10^5`

---

## Understanding BST Deletion

### Why is Deletion Complex?

Unlike insertion (always add at leaf), deletion has **three cases**:

#### Case 1: Node is a Leaf (No Children)
```
Delete 2:        Result:
      5              5
     / \            / \
    3   6          3   6
   /               
  2                

Simply remove the node!
```

#### Case 2: Node has One Child
```
Delete 3:        Result:
      5              5
     / \            / \
    3   6          2   6
   /               
  2                

Replace node with its child!
```

#### Case 3: Node has Two Children (Most Complex)
```
Delete 5:        Option 1 (Successor):    Option 2 (Predecessor):
      5                  6                        4
     / \                / \                      / \
    3   6              3   7                    3   6
   / \   \            / \                      /     \
  2   4   7          2   4                    2       7

Replace with inorder successor (smallest in right subtree)
OR inorder predecessor (largest in left subtree)
```

### Inorder Successor vs Predecessor

**Inorder Successor**: Smallest value in right subtree
- Go right once, then keep going left
- Guaranteed to have at most one child (right child)

**Inorder Predecessor**: Largest value in left subtree
- Go left once, then keep going right
- Guaranteed to have at most one child (left child)

```
Tree:           Successor of 5:    Predecessor of 5:
      5              6                   4
     / \            ↑                    ↑
    3   8          Go right,           Go left,
   / \  / \        then leftmost       then rightmost
  2  4 6   9
```

---

## Tree Node Definition

In [ ]:
class TreeNode:
    """Definition for a binary tree node."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values):
    """Helper function to build tree from list representation."""
    if not values:
        return None
    
    root = TreeNode(values[0])
    queue = [root]
    i = 1
    
    while queue and i < len(values):
        node = queue.pop(0)
        
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    
    return root

def print_tree(root, level=0, prefix="Root: "):
    """Helper function to visualize tree structure."""
    if root is not None:
        print(" " * (level * 4) + prefix + str(root.val))
        if root.left or root.right:
            if root.left:
                print_tree(root.left, level + 1, "L--- ")
            else:
                print(" " * ((level + 1) * 4) + "L--- None")
            if root.right:
                print_tree(root.right, level + 1, "R--- ")
            else:
                print(" " * ((level + 1) * 4) + "R--- None")

def tree_to_list(root):
    """Convert tree to list representation (level-order)."""
    if not root:
        return []
    
    result = []
    queue = [root]
    
    while queue:
        node = queue.pop(0)
        if node:
            result.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
        else:
            result.append(None)
    
    while result and result[-1] is None:
        result.pop()
    
    return result

def inorder_traversal(root):
    """Get inorder traversal (sorted for BST)."""
    if not root:
        return []
    return inorder_traversal(root.left) + [root.val] + inorder_traversal(root.right)

---

## Approach 1: Recursive Deletion (Inorder Successor)

### Intuition

The most common approach uses **inorder successor** for two-children case:
1. **Search**: Navigate to node to delete (like BST search)
2. **Delete**: Handle three cases:
   - No children: return null
   - One child: return that child
   - Two children: replace with inorder successor

### Algorithm

```
deleteNode(root, key):
    if root is null:
        return null
    
    # Search for node
    if key < root.val:
        root.left = deleteNode(root.left, key)
    elif key > root.val:
        root.right = deleteNode(root.right, key)
    else:
        # Found node to delete
        
        # Case 1 & 2: 0 or 1 child
        if not root.left:
            return root.right
        if not root.right:
            return root.left
        
        # Case 3: 2 children
        # Find inorder successor (smallest in right subtree)
        successor = findMin(root.right)
        root.val = successor.val
        root.right = deleteNode(root.right, successor.val)
    
    return root
```

### Complexity

- **Time**: O(h) where h is height
  - Best case (balanced): O(log n)
  - Worst case (skewed): O(n)
- **Space**: O(h) for recursion stack

In [ ]:
def deleteNode(root, key):
    """
    Delete node from BST using inorder successor.
    Time: O(h), Space: O(h)
    """
    if not root:
        return None
    
    # Search for node to delete
    if key < root.val:
        root.left = deleteNode(root.left, key)
    elif key > root.val:
        root.right = deleteNode(root.right, key)
    else:
        # Found node to delete
        
        # Case 1: No left child (includes leaf case)
        if not root.left:
            return root.right
        
        # Case 2: No right child
        if not root.right:
            return root.left
        
        # Case 3: Two children
        # Find inorder successor (smallest in right subtree)
        successor = root.right
        while successor.left:
            successor = successor.left
        
        # Replace current node's value with successor's value
        root.val = successor.val
        
        # Delete the successor (which has at most one child)
        root.right = deleteNode(root.right, successor.val)
    
    return root

# Test
test_cases = [
    ([5, 3, 6, 2, 4, None, 7], 3),
    ([5, 3, 6, 2, 4, None, 7], 0),
    ([5, 3, 6, 2, 4, None, 7], 5),
    ([2, 1], 2)
]

print("Recursive Deletion (Inorder Successor):\n")
for tree_vals, delete_key in test_cases:
    root = build_tree(tree_vals)
    print(f"Original tree: {tree_vals}")
    print(f"Delete key: {delete_key}")
    print("Before:")
    print_tree(root)
    
    result_root = deleteNode(root, delete_key)
    result = tree_to_list(result_root)
    
    print(f"\nResult: {result}")
    if result_root:
        print("After:")
        print_tree(result_root)
        print(f"Inorder (should be sorted): {inorder_traversal(result_root)}")
    print("\n" + "="*70 + "\n")

### Detailed Step-by-Step Trace

In [ ]:
def deleteNode_verbose(root, key, level=0, position="Root"):
    """Verbose deletion showing all three cases."""
    indent = "  " * level
    
    if not root:
        print(f"{indent}{position}: NULL (key {key} not found)")
        return None
    
    print(f"{indent}{position}: Node({root.val})")
    print(f"{indent}  Compare: {key} vs {root.val}")
    
    if key < root.val:
        print(f"{indent}  → {key} < {root.val}, search LEFT")
        root.left = deleteNode_verbose(root.left, key, level + 1, f"Left of {root.val}")
    elif key > root.val:
        print(f"{indent}  → {key} > {root.val}, search RIGHT")
        root.right = deleteNode_verbose(root.right, key, level + 1, f"Right of {root.val}")
    else:
        print(f"{indent}  → FOUND! Node to delete: {root.val}")
        
        # Determine case
        has_left = root.left is not None
        has_right = root.right is not None
        
        if not has_left and not has_right:
            print(f"{indent}  → CASE 1: Leaf node (no children)")
            print(f"{indent}  → Simply remove this node")
            return None
        elif not has_left:
            print(f"{indent}  → CASE 2: Only right child (Node {root.right.val})")
            print(f"{indent}  → Replace with right child")
            return root.right
        elif not has_right:
            print(f"{indent}  → CASE 2: Only left child (Node {root.left.val})")
            print(f"{indent}  → Replace with left child")
            return root.left
        else:
            print(f"{indent}  → CASE 3: Two children")
            print(f"{indent}  → Find inorder successor (smallest in right subtree)")
            
            # Find successor
            successor = root.right
            path = [successor.val]
            while successor.left:
                successor = successor.left
                path.append(successor.val)
            
            print(f"{indent}  → Path to successor: {' → '.join(map(str, path))}")
            print(f"{indent}  → Successor: {successor.val}")
            print(f"{indent}  → Replace {root.val} with {successor.val}")
            
            root.val = successor.val
            
            print(f"{indent}  → Now delete successor {successor.val} from right subtree")
            root.right = deleteNode_verbose(root.right, successor.val, level + 1, f"Right of {root.val}")
    
    print(f"{indent}  ↑ Return Node({root.val})")
    return root

# Test
print("Detailed Deletion Trace:\n")
print("Delete 3 from [5, 3, 6, 2, 4, None, 7]:\n")
print("Original tree:")
root = build_tree([5, 3, 6, 2, 4, None, 7])
print_tree(root)
print("\nDeletion trace:")
print("="*70)
result = deleteNode_verbose(root, 3)
print("="*70)
print("\nFinal tree:")
print_tree(result)

---

## Approach 2: Using Inorder Predecessor

### Intuition

Alternative approach using **inorder predecessor** for two-children case:
- Find largest value in left subtree
- Replace node with predecessor
- Delete predecessor from left subtree

### Complexity

- **Time**: O(h)
- **Space**: O(h)

In [ ]:
def deleteNode_predecessor(root, key):
    """
    Delete node from BST using inorder predecessor.
    Time: O(h), Space: O(h)
    """
    if not root:
        return None
    
    if key < root.val:
        root.left = deleteNode_predecessor(root.left, key)
    elif key > root.val:
        root.right = deleteNode_predecessor(root.right, key)
    else:
        # Found node to delete
        
        if not root.left:
            return root.right
        if not root.right:
            return root.left
        
        # Two children: use predecessor
        # Find largest in left subtree
        predecessor = root.left
        while predecessor.right:
            predecessor = predecessor.right
        
        root.val = predecessor.val
        root.left = deleteNode_predecessor(root.left, predecessor.val)
    
    return root

# Test
print("Deletion Using Inorder Predecessor:\n")
root = build_tree([5, 3, 6, 2, 4, None, 7])
print("Original tree:")
print_tree(root)
print("\nDelete 3:")
result = deleteNode_predecessor(root, 3)
print("\nResult tree:")
print_tree(result)
print(f"\nInorder: {inorder_traversal(result)}")

---

## Approach 3: Iterative Deletion

### Intuition

Avoid recursion by using iteration:
1. Find node to delete and its parent
2. Handle three cases
3. Update parent's pointer

### Complexity

- **Time**: O(h)
- **Space**: O(1)

In [ ]:
def deleteNode_iterative(root, key):
    """
    Iterative deletion from BST.
    Time: O(h), Space: O(1)
    """
    # Find node and its parent
    parent = None
    current = root
    
    while current and current.val != key:
        parent = current
        if key < current.val:
            current = current.left
        else:
            current = current.right
    
    # Key not found
    if not current:
        return root
    
    # Determine replacement node
    if not current.left:
        replacement = current.right
    elif not current.right:
        replacement = current.left
    else:
        # Two children: find successor
        successor_parent = current
        successor = current.right
        
        while successor.left:
            successor_parent = successor
            successor = successor.left
        
        # Replace current's value with successor's
        current.val = successor.val
        
        # Remove successor
        if successor_parent == current:
            successor_parent.right = successor.right
        else:
            successor_parent.left = successor.right
        
        return root
    
    # Update parent's pointer
    if not parent:
        return replacement
    
    if parent.left == current:
        parent.left = replacement
    else:
        parent.right = replacement
    
    return root

# Test
print("Iterative Deletion:\n")
root = build_tree([5, 3, 6, 2, 4, None, 7])
print("Original tree:")
print_tree(root)
print("\nDelete 3:")
result = deleteNode_iterative(root, 3)
print("\nResult tree:")
print_tree(result)
print(f"\nInorder: {inorder_traversal(result)}")

---

## Visualizing All Three Cases

In [ ]:
print("Demonstrating All Three Deletion Cases:\n")
print("="*70)

# Case 1: Delete leaf
print("CASE 1: Delete Leaf Node\n")
root1 = build_tree([5, 3, 6, 2, 4, None, 7])
print("Original:")
print_tree(root1)
print("\nDelete 2 (leaf):")
result1 = deleteNode(root1, 2)
print_tree(result1)
print(f"Inorder: {inorder_traversal(result1)}")
print("\n" + "="*70 + "\n")

# Case 2: Delete node with one child
print("CASE 2: Delete Node with One Child\n")
root2 = build_tree([5, 3, 6, None, 4, None, 7])
print("Original:")
print_tree(root2)
print("\nDelete 3 (has only right child 4):")
result2 = deleteNode(root2, 3)
print_tree(result2)
print(f"Inorder: {inorder_traversal(result2)}")
print("\n" + "="*70 + "\n")

# Case 3: Delete node with two children
print("CASE 3: Delete Node with Two Children\n")
root3 = build_tree([5, 3, 6, 2, 4, None, 7])
print("Original:")
print_tree(root3)
print("\nDelete 3 (has two children: 2 and 4):")
print("Inorder successor of 3 is 4 (smallest in right subtree)")
result3 = deleteNode(root3, 3)
print("\nAfter deletion:")
print_tree(result3)
print(f"Inorder: {inorder_traversal(result3)}")
print("\n" + "="*70)

---

## Edge Cases and Special Scenarios

In [ ]:
print("Edge Cases:\n")

# Edge case 1: Delete root
print("1. Delete root node:")
root = build_tree([5, 3, 6, 2, 4, None, 7])
print("   Before:")
print_tree(root)
result = deleteNode(root, 5)
print("   After deleting 5:")
print_tree(result)
print(f"   Inorder: {inorder_traversal(result)}\n")

# Edge case 2: Delete from single node tree
print("2. Delete from single node tree:")
root = build_tree([5])
print("   Before:")
print_tree(root)
result = deleteNode(root, 5)
print(f"   After deleting 5: {tree_to_list(result)}\n")

# Edge case 3: Delete non-existent key
print("3. Delete non-existent key:")
root = build_tree([5, 3, 6, 2, 4, None, 7])
print("   Before:")
print_tree(root)
result = deleteNode(root, 10)
print("   After trying to delete 10 (doesn't exist):")
print_tree(result)
print(f"   Tree unchanged: {tree_to_list(result)}\n")

# Edge case 4: Delete from empty tree
print("4. Delete from empty tree:")
root = None
result = deleteNode(root, 5)
print(f"   Result: {tree_to_list(result)}\n")

# Edge case 5: Delete all nodes one by one
print("5. Delete all nodes one by one:")
root = build_tree([5, 3, 6, 2, 4, None, 7])
values = [2, 4, 3, 7, 6, 5]
print(f"   Starting tree: {tree_to_list(root)}")
print(f"   Delete order: {values}")
for val in values:
    root = deleteNode(root, val)
    print(f"   After deleting {val}: {tree_to_list(root)}")
print(f"   Final: {tree_to_list(root)} (empty)")

---

## Comparison of Approaches

| Approach | Time | Space | Pros | Cons |
|----------|------|-------|------|------|
| **Recursive (Successor)** | O(h) | O(h) | Clean, intuitive | Stack overflow risk |
| **Recursive (Predecessor)** | O(h) | O(h) | Alternative valid solution | Stack overflow risk |
| **Iterative** | O(h) | O(1) | Space-efficient, no recursion | More complex code |

### Successor vs Predecessor

Both are valid! The choice doesn't affect correctness:

**Inorder Successor** (more common):
- Go right, then leftmost
- Smallest value greater than current

**Inorder Predecessor**:
- Go left, then rightmost
- Largest value smaller than current

```
Delete 5:       Successor (6):      Predecessor (4):
      5              6                     4
     / \            / \                   / \
    3   6          3   7                 3   6
   / \   \        / \                   /     \
  2   4   7      2   4                 2       7

Both maintain BST property!
```

---

## Why Deletion is Complex

### The Challenge

Unlike insertion (always add at leaf), deletion must:
1. **Find** the node
2. **Handle** three different cases
3. **Maintain** BST property
4. **Preserve** all other nodes

### The Three Cases Explained

#### Why Case 1 (Leaf) is Easy
```
No children → just remove
No other nodes affected
```

#### Why Case 2 (One Child) is Easy
```
One child → promote that child
Child's subtree maintains BST property
```

#### Why Case 3 (Two Children) is Hard
```
Two children → can't just remove
Need to find replacement that:
  1. Maintains BST property
  2. Fits in current position
  3. Is easy to remove

Successor/Predecessor guarantees:
  ✓ Maintains BST property
  ✓ Has at most one child
  ✓ Easy to remove (Case 1 or 2)
```

### Why Successor Works

```
Original:           After replacing with successor:
      5                      6
     / \                    / \
    3   8                  3   8
   / \  / \               / \   \
  2  4 6   9             2  4    9
        \
         7

Successor (6) is:
  ✓ Greater than all in left subtree (3, 2, 4)
  ✓ Smaller than all in right subtree (8, 7, 9)
  ✓ Has at most one child (right child 7)
```

---

## BST Operations Summary

### Complete BST Operations

Now we've covered all three main BST operations:

#### 1. Search (LeetCode 700)
```python
def search(root, val):
    if not root or root.val == val:
        return root
    if val < root.val:
        return search(root.left, val)
    return search(root.right, val)
```
**Complexity**: O(h)

#### 2. Insert (LeetCode 701)
```python
def insert(root, val):
    if not root:
        return TreeNode(val)
    if val < root.val:
        root.left = insert(root.left, val)
    else:
        root.right = insert(root.right, val)
    return root
```
**Complexity**: O(h)

#### 3. Delete (LeetCode 450) ← This Problem
```python
def delete(root, val):
    if not root:
        return None
    if val < root.val:
        root.left = delete(root.left, val)
    elif val > root.val:
        root.right = delete(root.right, val)
    else:
        if not root.left: return root.right
        if not root.right: return root.left
        # Two children: use successor
        successor = findMin(root.right)
        root.val = successor.val
        root.right = delete(root.right, successor.val)
    return root
```
**Complexity**: O(h)

### Related Problems

**BST Operations:**
- [700. Search in a Binary Search Tree](https://leetcode.com/problems/search-in-a-binary-search-tree/)
- [701. Insert into a Binary Search Tree](https://leetcode.com/problems/insert-into-a-binary-search-tree/)
- [450. Delete Node in a BST](https://leetcode.com/problems/delete-node-in-a-bst/) ← This problem

**BST Properties:**
- [98. Validate Binary Search Tree](https://leetcode.com/problems/validate-binary-search-tree/)
- [230. Kth Smallest Element in a BST](https://leetcode.com/problems/kth-smallest-element-in-a-bst/)
- [235. Lowest Common Ancestor of a BST](https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-search-tree/)

**BST Construction:**
- [108. Convert Sorted Array to Binary Search Tree](https://leetcode.com/problems/convert-sorted-array-to-binary-search-tree/)
- [109. Convert Sorted List to Binary Search Tree](https://leetcode.com/problems/convert-sorted-list-to-binary-search-tree/)

---

## Key Takeaways

### Understanding BST Deletion

1. **Three cases to handle**:
   - Leaf (no children): Simply remove
   - One child: Replace with that child
   - Two children: Replace with successor/predecessor

2. **Why successor/predecessor works**:
   - Maintains BST property
   - Has at most one child
   - Easy to remove (becomes Case 1 or 2)

3. **Inorder successor**: Smallest in right subtree
   - Go right once, then leftmost

4. **Inorder predecessor**: Largest in left subtree
   - Go left once, then rightmost

### Algorithm Strategy

1. **Search phase**: Navigate to node (like BST search)

2. **Delete phase**: Handle based on children count

3. **Maintain BST**: Ensure property holds after deletion

### Three Approaches

1. **Recursive (Successor)** - Most common
   - Clean, intuitive
   - O(h) space for recursion

2. **Recursive (Predecessor)** - Alternative
   - Equally valid
   - Different replacement choice

3. **Iterative** - Space-efficient
   - O(1) space
   - More complex logic

### Complexity Analysis

- **Time**: O(h) for all approaches
  - Best: O(log n) - balanced tree
  - Worst: O(n) - skewed tree

- **Space**:
  - Recursive: O(h) - recursion stack
  - Iterative: O(1) - no extra space

### Common Mistakes

1. **Forgetting to return updated root**
   - Always return root after modification

2. **Not handling all three cases**
   - Must check: 0, 1, or 2 children

3. **Incorrect successor/predecessor**
   - Successor: right then leftmost
   - Predecessor: left then rightmost

4. **Not deleting successor/predecessor**
   - After copying value, must delete original

### Remember

🎯 **Three cases** - leaf, one child, two children  
🎯 **Successor** - smallest in right subtree  
🎯 **Predecessor** - largest in left subtree  
🎯 **Replace & Delete** - copy value, then delete original  
🎯 **O(h) complexity** - height determines efficiency  

Master BST deletion and you'll understand one of the most complex tree operations!